# 2-4のデータ取得のための付録
- このプログラムは、東京における過去の気象データ（1980年〜2025年8月の日平均気温）を、Web API から取得し、CSV ファイルとして保存する。
- Open-Meteo という無料の気象データ APIを利用する。
-  APIとは「アプリケーション・プログラミング・インターフェース（Application Programming Interface）」の略称であり、あるソフトウェアの機能を他のソフトウェアから呼び出す仕組み。

## 1. 必要なライブラリの読み込み

**requests**: Web 上の API（URL）にアクセスし、データを取得するためのライブラリ。

In [55]:
import requests
import pandas as pd

## 2. 東京の位置情報を指定する

In [56]:
# 東京の緯度・経度
lat = 35.6895
lon = 139.6917

## 3. API に渡す条件（パラメータ）をまとめる

Open-Meteo の履歴気象データ API に対して「どの地点の・いつからいつまでの・どの気象要素を取得するか」を指定するためのパラメータ一とその設定値をparamsに代入。


| パラメータ名 | 設定値 | 意味・役割 |
|---|---|---|
| `latitude` | `35.6895` | 取得地点の**緯度**（東京付近の代表値） |
| `longitude` | `139.6917` | 取得地点の**経度**（東京付近の代表値） |
| `start_date` | `"1980-08-01"` | データ取得の**開始日**（YYYY-MM-DD） |
| `end_date` | `"2025-08-31"` | データ取得の**終了日**（YYYY-MM-DD） |
| `daily` | `"temperature_2m_mean"` | 日別に取得する変数：**地上2 m の日平均気温（℃）** |
| `timezone` | `"Asia/Tokyo"` | 日付を**日本時間（JST）**で扱うための設定 |


In [57]:
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "1980-08-01",
    "end_date": "2025-08-31",
    "daily": "temperature_2m_mean",
    "timezone": "Asia/Tokyo"
}

## 4. Web API にアクセスしてデータを取得する

- **requests.get()**: 指定した URL にアクセスし、データを取得する。
- **params = params**: 上で設定した条件を URL に付加する。
- **timeout = 60**: リクエストから60秒経過しても応答がなければ強制的にリクエストを中止する。
- **raise_for_status()**: 通信エラーが起きた場合に、分かりやすくエラーを出すための処理。

In [58]:
url = "https://archive-api.open-meteo.com/v1/archive"
r = requests.get(url, params=params, timeout=60)
r.raise_for_status()

## 5. 取得したデータを Python の辞書として読み込む（辞書については後日改めて説明）

In [59]:
data = r.json()

## 6. pandas の DataFrame に変換する

- **date**: 日付
- **tmean_C**: 日平均気温（℃）

の2列から成る表形式のデータ（DataFrame）をdfとして作成。

In [60]:
# DataFrame 化
df = pd.DataFrame({
    "date": data["daily"]["time"],
    "tmean_C": data["daily"]["temperature_2m_mean"]
})
print(df)

             date  tmean_C
0      1980-01-01      3.8
1      1980-01-02      1.8
2      1980-01-03      4.5
3      1980-01-04      8.5
4      1980-01-05      2.8
...           ...      ...
16797  2025-12-27      2.5
16798  2025-12-28      3.6
16799  2025-12-29      5.2
16800  2025-12-30      7.2
16801  2025-12-31      6.9

[16802 rows x 2 columns]


## 7. 日付を「日付型」に変換する

文字列のままだと月ごとの抽出や並び替えがしにくいので、to_datetime を使うことで、日付の扱いを容易にする。

- **.dt.year**: 年を取り出す
- **.dt.month**: 月を取り出す
- **.dt.day**: 日を取り出す

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [50]:
print(df["date"])
print(df["date"].dt.year)
print(df["date"].dt.month)
print(df["date"].dt.day)

0       1980-08-01
1       1980-08-02
2       1980-08-03
3       1980-08-04
4       1980-08-05
           ...    
16462   2025-08-27
16463   2025-08-28
16464   2025-08-29
16465   2025-08-30
16466   2025-08-31
Name: date, Length: 16467, dtype: datetime64[ns]
0        1980
1        1980
2        1980
3        1980
4        1980
         ... 
16462    2025
16463    2025
16464    2025
16465    2025
16466    2025
Name: date, Length: 16467, dtype: int32
0        8
1        8
2        8
3        8
4        8
        ..
16462    8
16463    8
16464    8
16465    8
16466    8
Name: date, Length: 16467, dtype: int32
0         1
1         2
2         3
3         4
4         5
         ..
16462    27
16463    28
16464    29
16465    30
16466    31
Name: date, Length: 16467, dtype: int32


## 8. 8月のデータだけを抽出する

**(df["date"].dt.month == 8)**: dfのうち月が8と等しいものだけを指定

In [53]:
df = df[(df["date"].dt.month == 8)]

## 9. CSV ファイルとして保存する

- **index=False**: 行番号を CSV に書き出さない
- **encoding="utf-8-sig"**: Excel で日本語が文字化けしにくい文字コード

In [61]:
# CSV 出力
out_file = "tokyo_daily_mean_temperature_aug_1980_2025.csv"
df.to_csv(out_file, index=False, encoding="utf-8-sig")

print(f"Saved: {out_file}")
print(df.head())
print(df.tail())

Saved: tokyo_daily_mean_temperature_1980_2025.csv
         date  tmean_C
0  1980-01-01      3.8
1  1980-01-02      1.8
2  1980-01-03      4.5
3  1980-01-04      8.5
4  1980-01-05      2.8
             date  tmean_C
16797  2025-12-27      2.5
16798  2025-12-28      3.6
16799  2025-12-29      5.2
16800  2025-12-30      7.2
16801  2025-12-31      6.9
